# 04. Structured Outputs：Grammar、Token Mask 与 Compressed FSM

> 面向即将加入 SGLang 开源社区的开发者。建议从仓库根目录启动 Jupyter，
> 或者在 notebook 第一格运行路径检查。本文以本地源码为主，线上文档为索引。
> Markdown 中的源码摘录会插入 `# 注：...` 行内讲解；可执行代码 cell 则保持可运行。


In [ ]:
from pathlib import Path
import ast, inspect, re, textwrap

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "python" / "sglang").exists() and (candidate / "docs").exists():
            return candidate
    raise RuntimeError("没有找到 SGLang 仓库根目录，请从仓库内启动 notebook。")

ROOT = find_repo_root()
print(ROOT)

def read_rel(path):
    return (ROOT / path).read_text()

def show_lines(path, start, end):
    lines = read_rel(path).splitlines()
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:4d}: {lines[i-1]}")

def find_defs(path, names=None):
    tree = ast.parse(read_rel(path))
    rows = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            if names is None or node.name in names:
                rows.append((node.lineno, type(node).__name__, node.name))
    return sorted(rows)


## 从用户视角到实现视角

用户传入 JSON schema、regex、EBNF 或 structural tag。实现上，SGLang 要在每一步采样前知道：

> 当前 grammar 状态下，词表中哪些 token 仍然合法？

因此 structured outputs 的核心不是“生成 JSON 字符串”，而是：

1. 把约束编译成 grammar object / matcher / FSM。
2. 每步根据当前状态填充 token bitmask。
3. 在 logits 上应用 mask。
4. 采样 token 后调用 `accept_token` 推进 grammar 状态。
5. 对确定性片段尝试 jump-forward，跳过逐 token 解码。


In [ ]:
for path in [
    "python/sglang/srt/constrained/base_grammar_backend.py",
    "python/sglang/srt/constrained/grammar_manager.py",
    "python/sglang/srt/constrained/xgrammar_backend.py",
    "python/sglang/srt/constrained/outlines_backend.py",
    "python/sglang/srt/constrained/outlines_jump_forward.py",
]:
    print("\n==", path)
    for row in find_defs(path, names={"BaseGrammarObject", "BaseGrammarBackend", "GrammarManager", "XGrammarGrammar", "XGrammarGrammarBackend", "OutlinesJumpForwardMap"}):
        print(row)


## `BaseGrammarObject` 是采样循环看到的接口

无论后端是 xgrammar、llguidance 还是 outlines，scheduler/sampling 层关心的接口是一组共同方法：

- `fill_vocab_mask`
- `apply_vocab_mask`
- `accept_token`
- `rollback`
- `try_jump_forward`
- `jump_and_retokenize`

这也是新增 grammar backend 时最重要的 contract。


In [ ]:
show_lines("python/sglang/srt/constrained/base_grammar_backend.py", 30, 115)


### `BaseGrammarObject`：采样循环只依赖这组状态机方法

```python
# python/sglang/srt/constrained/base_grammar_backend.py:30-113
  30: @dataclass
  31: class GrammarStats:
      # 注：类定义：`GrammarStats` 是这一段的状态/行为边界；先看字段，再看哪些方法会改字段。
  32:     compilation_time: Optional[float] = None
      # 注：字段/变量声明：`compilation_time` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
  33:     schema_count: Optional[int] = None
      # 注：字段/变量声明：`schema_count` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
  34:     ebnf_size: Optional[int] = None
      # 注：字段/变量声明：`ebnf_size` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
  35:     is_cache_hit: bool = False
      # 注：字段/变量声明：`is_cache_hit` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
  36:     is_grammar_aborted: bool = False
      # 注：字段/变量声明：`is_grammar_aborted` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
  37:     tree_traversal_time: List[float] = field(default_factory=list)
      # 注：字段/变量声明：`tree_traversal_time` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
  38:     dispatch_type: Optional[str] = None
      # 注：字段/变量声明：`dispatch_type` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
  39:     num_timeout: int = 0
      # 注：字段/变量声明：`num_timeout` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
  40: 
  41: 
  42: class BaseGrammarObject:
      # 注：类定义：`BaseGrammarObject` 是这一段的状态/行为边界；先看字段，再看哪些方法会改字段。
  43: 
  44:     def __init__(self):
      # 注：函数定义：`__init__` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  45:         self._finished = False
      # 注：成员变量写入：`self._finished` 会留在对象生命周期中，后续方法可能依赖这份状态。
  46:         self.grammar_stats = None
      # 注：成员变量写入：`self.grammar_stats` 会留在对象生命周期中，后续方法可能依赖这份状态。
  47:         self.current_token = None
      # 注：成员变量写入：`self.current_token` 保存 token 相关状态，后续长度计算、cache key 或采样都会读取它。
  48: 
  49:     def maybe_init_reasoning(self, reasoning: bool):
      # 注：函数定义：`maybe_init_reasoning` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  50:         pass
  51: 
  52:     def accept_token(self, token: int) -> None:
      # 注：函数定义：`accept_token` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  53:         """
  54:         Accept a token in the grammar.
  55:         """
  56:         raise NotImplementedError()
      # 注：函数/库调用：`NotImplementedError` 把当前阶段委托给外部 helper 或库实现。
  57: 
  58:     def rollback(self, k: int):
      # 注：函数定义：`rollback` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  59:         raise NotImplementedError()
      # 注：函数/库调用：`NotImplementedError` 把当前阶段委托给外部 helper 或库实现。
  60: 
  61:     def is_terminated(self):
      # 注：函数定义：`is_terminated` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  62:         return False
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
  63: 
  64:     def allocate_vocab_mask(
      # 注：函数定义：`allocate_vocab_mask` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  65:         self, vocab_size: int, batch_size: int, device
  66:     ) -> torch.Tensor:
  67:         raise NotImplementedError()
      # 注：函数/库调用：`NotImplementedError` 把当前阶段委托给外部 helper 或库实现。
  68: 
  69:     def fill_vocab_mask(self, vocab_mask: torch.Tensor, idx: int) -> None:
      # 注：函数定义：`fill_vocab_mask` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  70:         raise NotImplementedError()
      # 注：函数/库调用：`NotImplementedError` 把当前阶段委托给外部 helper 或库实现。
  71: 
  72:     @staticmethod
  73:     def move_vocab_mask(vocab_mask: torch.Tensor, device) -> torch.Tensor:
      # 注：函数定义：`move_vocab_mask` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  74:         raise NotImplementedError()
      # 注：函数/库调用：`NotImplementedError` 把当前阶段委托给外部 helper 或库实现。
  75: 
  76:     @staticmethod
  77:     def apply_vocab_mask(logits: torch.Tensor, vocab_mask: torch.Tensor) -> None:
      # 注：函数定义：`apply_vocab_mask` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  78:         raise NotImplementedError()
      # 注：函数/库调用：`NotImplementedError` 把当前阶段委托给外部 helper 或库实现。
  79: 
  80:     def copy(self) -> "BaseGrammarObject":
      # 注：函数定义：`copy` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  81:         return self
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
  82: 
  83:     @property
  84:     def finished(self):
      # 注：函数定义：`finished` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  85:         return self._finished
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
  86: 
  87:     @finished.setter
  88:     def finished(self, finished):
      # 注：函数定义：`finished` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  89:         self._finished = finished
      # 注：成员变量写入：`self._finished` 会留在对象生命周期中，后续方法可能依赖这份状态。
  90: 
  91:     def try_jump_forward(self, tokenizer) -> Optional[Tuple[List[int], str]]:
      # 注：函数定义：`try_jump_forward` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  92:         """
  93:         Try to jump forward in the grammar.
  94: 
  95:         Returns:
  96:             A jump forward helper which may be used in `jump_forward_str_state`.
  97:             None if the jump forward is not possible.
  98:         """
  99:         raise NotImplementedError()
      # 注：函数/库调用：`NotImplementedError` 把当前阶段委托给外部 helper 或库实现。
 100: 
 101:     def jump_forward_str_state(self, helper: Tuple[List[int], str]) -> Tuple[str, int]:
      # 注：函数定义：`jump_forward_str_state` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 102:         """
 103:         Jump forward for the grammar.
 104: 
 105:         Returns:
 106:             A tuple of the jump forward string and the next state of the grammar
 107:             (which can be used in `jump_and_retokenize` if needed).
 108:         """
 109:         raise NotImplementedError()
      # 注：函数/库调用：`NotImplementedError` 把当前阶段委托给外部 helper 或库实现。
 110: 
 111:     def jump_and_retokenize(
      # 注：函数定义：`jump_and_retokenize` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 112:         self, old_output_ids: List[int], new_output_ids: List[int], next_state: int
 113:     ) -> None:
```

**读这段时抓住：**

- `accept_token` 是状态机前进；采样出 token 后必须调用它，否则下一步 mask 会停在旧状态。
- `allocate_vocab_mask` / `fill_vocab_mask` / `apply_vocab_mask` 把 grammar 状态转成 logits 过滤。
- `rollback` 是 speculative/jump-forward/retokenize 场景的保险丝，允许撤回若干 token 后重放。
- `try_jump_forward` 和 `jump_and_retokenize` 是 compressed FSM 加速的接口，不是所有 backend 都必须成功返回。


## GrammarManager：异步编译与队列

Grammar 编译可能比普通请求准备更慢。SGLang 不会让未编译完成的请求直接进入 waiting queue，而是放入 `grammar_queue`。编译结果会缓存；多 rank 下还要同步哪些请求 ready/failed。


In [ ]:
show_lines("python/sglang/srt/constrained/grammar_manager.py", 55, 145)


### `GrammarManager.process_req_with_grammar`：请求什么时候被拦在 grammar queue

```python
# python/sglang/srt/constrained/grammar_manager.py:83-134
  83:         thinking_budget = self._get_request_thinking_budget(req)
      # 注：局部状态绑定：`thinking_budget` 保存本阶段的中间结果，后续几行通常会立即消费它。
  84:         if thinking_budget is None:
      # 注：分支判断：只有满足 `thinking_budget is None` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
  85:             return
      # 注：返回出口：提前结束当前路径，通常表示这个分支已经完成处理或无需继续推进。
  86:         if isinstance(req.grammar, ReasonerGrammarObject):
      # 注：分支判断：根据请求的采样/grammar 约束 `isinstance(req.grammar, ReasonerGrammarObject)` 决定是否进入受限解码路径。
  87:             req.grammar.max_think_tokens = thinking_budget
      # 注：对象状态写入：`req.grammar.max_think_tokens` 修改 `req` 的可见状态，读源码时要继续追踪它在哪里被消费。
  88: 
  89:     def process_req_with_grammar(self, req: Req) -> bool:
      # 注：函数定义：`process_req_with_grammar` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  90:         # Init grammar cache for this request
  91:         add_to_grammar_queue = False
      # 注：局部状态绑定：`add_to_grammar_queue` 保存本阶段的中间结果，后续几行通常会立即消费它。
  92:         if (
      # 注：多行分支开始：完整条件在接下来几行，通常用于组合多个请求参数或运行时状态。
  93:             req.sampling_params.json_schema is not None
  94:             or req.sampling_params.regex is not None
  95:             or req.sampling_params.ebnf is not None
  96:             or req.sampling_params.structural_tag is not None
  97:         ):
  98:             if self.grammar_backend is None:
      # 注：分支判断：根据请求的采样/grammar 约束 `self.grammar_backend is None` 决定是否进入受限解码路径。
  99:                 error_msg = "Grammar-based generation (json_schema, regex, ebnf, structural_tag) is not supported when the server is launched with --grammar-backend none"
      # 注：局部状态绑定：`error_msg` 保存本阶段的中间结果，后续几行通常会立即消费它。
 100:                 req.set_finish_with_abort(error_msg)
      # 注：对象/库方法调用：`req.set_finish_with_abort` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 101:             else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
 102:                 if req.sampling_params.json_schema is not None:
      # 注：分支判断：根据请求的采样/grammar 约束 `req.sampling_params.json_schema is not None` 决定是否进入受限解码路径。
 103:                     key = ("json", req.sampling_params.json_schema)
      # 注：局部状态绑定：`key` 保存本阶段的中间结果，后续几行通常会立即消费它。
 104:                 elif req.sampling_params.regex is not None:
      # 注：补充分支：前面的条件不满足时，再检查 `req.sampling_params.regex is not None`。
 105:                     key = ("regex", req.sampling_params.regex)
      # 注：局部状态绑定：`key` 保存本阶段的中间结果，后续几行通常会立即消费它。
 106:                 elif req.sampling_params.ebnf is not None:
      # 注：补充分支：前面的条件不满足时，再检查 `req.sampling_params.ebnf is not None`。
 107:                     key = ("ebnf", req.sampling_params.ebnf)
      # 注：局部状态绑定：`key` 保存本阶段的中间结果，后续几行通常会立即消费它。
 108:                 elif req.sampling_params.structural_tag:
      # 注：补充分支：前面的条件不满足时，再检查 `req.sampling_params.structural_tag`。
 109:                     key = ("structural_tag", req.sampling_params.structural_tag)
      # 注：局部状态绑定：`key` 保存本阶段的中间结果，后续几行通常会立即消费它。
 110: 
 111:                 value, cache_hit = self.grammar_backend.get_cached_or_future_value(
      # 注：局部状态绑定：`value, cache_hit` 接住关键调用结果，后续代码会基于它继续装配组件或推进请求。
 112:                     key, req.require_reasoning
 113:                 )
 114:                 req.grammar = value
      # 注：对象状态写入：`req.grammar` 修改 `req` 的可见状态，读源码时要继续追踪它在哪里被消费。
 115: 
 116:                 if not cache_hit:
      # 注：分支判断：根据缓存/内存资源 `not cache_hit` 决定是否复用、加载、驱逐或降级。
 117:                     req.grammar_key = key
      # 注：对象状态写入：`req.grammar_key` 修改 `req` 的可见状态，读源码时要继续追踪它在哪里被消费。
 118:                     add_to_grammar_queue = True
      # 注：局部状态绑定：`add_to_grammar_queue` 保存本阶段的中间结果，后续几行通常会立即消费它。
 119:                 else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
 120:                     if isinstance(
      # 注：分支判断：只有满足 `isinstance(` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 121:                         value, InvalidGrammarObject
 122:                     ):  # We hit a cached invalid grammar.
 123:                         error_msg = (
      # 注：局部状态绑定：`error_msg` 保存本阶段的中间结果，后续几行通常会立即消费它。
 124:                             f"Failed to compile {key[0]} grammar: {value.error_message}"
 125:                         )
 126:                         req.set_finish_with_abort(error_msg)
      # 注：对象/库方法调用：`req.set_finish_with_abort` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 127:                     else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
 128:                         self._apply_request_reasoning_budget(req)
      # 注：成员函数调用：`self._apply_request_reasoning_budget` 会读取或更新当前对象状态，建议继续查看该方法定义。
 129:         elif self._enable_strict_thinking:
      # 注：补充分支：前面的条件不满足时，再检查 `self._enable_strict_thinking`。
 130:             grammar_obj = self.grammar_backend.init_strict_reasoning_grammar(
      # 注：局部状态绑定：`grammar_obj` 保存本阶段的中间结果，后续几行通常会立即消费它。
 131:                 req.require_reasoning
 132:             )
 133:             if grammar_obj is not None:
      # 注：分支判断：根据请求的采样/grammar 约束 `grammar_obj is not None` 决定是否进入受限解码路径。
 134:                 req.grammar = grammar_obj
      # 注：对象状态写入：`req.grammar` 修改 `req` 的可见状态，读源码时要继续追踪它在哪里被消费。
```

**读这段时抓住：**

- 只要 sampling params 中有 JSON schema、regex、EBNF 或 structural tag，就会构造 cache key。
- `get_cached_or_future_value` 可能返回现成 grammar，也可能返回 Future；Future 未完成时请求不能进入普通 waiting queue。
- invalid grammar 会被缓存为 `InvalidGrammarObject`，避免同一个坏 schema 反复编译。
- strict thinking 复用同一套 grammar/token filter 机制，所以这里还处理 reasoning budget。


### `GrammarManager.get_ready_grammar_requests`：多 rank 下 ready/failed 要同步

```python
# python/sglang/srt/constrained/grammar_manager.py:136-212
 136: 
 137:         if add_to_grammar_queue:
      # 注：分支判断：根据请求的采样/grammar 约束 `add_to_grammar_queue` 决定是否进入受限解码路径。
 138:             self.grammar_queue.append(req)
      # 注：成员函数调用：`self.grammar_queue.append` 会读取或更新当前对象状态，建议继续查看该方法定义。
 139: 
 140:         return add_to_grammar_queue
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
 141: 
 142:     def get_ready_grammar_requests(self) -> List[Req]:
      # 注：函数定义：`get_ready_grammar_requests` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 143:         """
 144:         Move requests whose grammar objects are ready from grammar_queue to waiting_queue.
 145: 
 146:         Rank i returns two sets ready_reqs_i, failed_reqs_i
 147:         ready_reqs_all = all_gather(ready_reqs_i)
      # 注：局部状态绑定：`ready_reqs_all` 保存请求或 batch 状态，后续调度/执行会继续改写它。
 148:         failed_reqs_all = all_gather(failed_reqs_i)
      # 注：局部状态绑定：`failed_reqs_all` 保存请求或 batch 状态，后续调度/执行会继续改写它。
 149: 
 150:         ready_reqs = intersect(ready_reqs_all)
      # 注：局部状态绑定：`ready_reqs` 保存请求或 batch 状态，后续调度/执行会继续改写它。
 151:         failed_reqs = union(failed_reqs_all)
      # 注：局部状态绑定：`failed_reqs` 保存请求或 batch 状态，后续调度/执行会继续改写它。
 152:         """
 153:         assert self.grammar_backend
 154:         ready_req_idxs: set[int] = set()
      # 注：字段/变量声明：`ready_req_idxs` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 155:         failed_req_idxs: set[int] = set()
      # 注：字段/变量声明：`failed_req_idxs` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 156: 
 157:         # Poll for ready requests
 158:         start_time = time.perf_counter()
      # 注：局部状态绑定：`start_time` 保存本阶段的中间结果，后续几行通常会立即消费它。
 159:         while time.perf_counter() - start_time < self.SGLANG_GRAMMAR_POLL_INTERVAL:
      # 注：循环等待/轮询：注意退出条件，否则容易造成 busy wait 或阻塞。
 160:             for i, req in enumerate(self.grammar_queue):
      # 注：循环处理：通常是在遍历请求、rank、worker、token 或候选项。
 161:                 if i in ready_req_idxs:
      # 注：分支判断：根据请求/batch 状态 `i in ready_req_idxs` 决定调度、执行或回包路径。
 162:                     continue
      # 注：循环控制：跳过或结束当前循环轮次，通常代表这个请求/节点已被当前分支处理完。
 163: 
 164:                 if req.finished() or req.grammar is None:  # It is aborted by AbortReq
      # 注：分支判断：根据请求的采样/grammar 约束 `req.finished() or req.grammar is None:  # It is aborted by AbortReq` 决定是否进入受限解码路径。
 165:                     ready_req_idxs.add(i)
      # 注：对象/库方法调用：`ready_req_idxs.add` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 166:                     continue
      # 注：循环控制：跳过或结束当前循环轮次，通常代表这个请求/节点已被当前分支处理完。
 167: 
 168:                 assert isinstance(req.grammar, futures.Future), f"{req=}"
 169:                 if req.grammar.done():
      # 注：分支判断：根据请求的采样/grammar 约束 `req.grammar.done()` 决定是否进入受限解码路径。
 170:                     ready_req_idxs.add(i)
      # 注：对象/库方法调用：`ready_req_idxs.add` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 171: 
 172:             # Sleep a bit to avoid busy waiting
 173:             time.sleep(self.SGLANG_GRAMMAR_POLL_INTERVAL / 10)
      # 注：对象/库方法调用：`time.sleep` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 174: 
 175:         # Check failed requests
 176:         for i, req in enumerate(self.grammar_queue):
      # 注：循环处理：通常是在遍历请求、rank、worker、token 或候选项。
 177:             if i not in ready_req_idxs:
      # 注：分支判断：根据请求/batch 状态 `i not in ready_req_idxs` 决定调度、执行或回包路径。
 178:                 self.grammar_queue[i].grammar_wait_ct += 1
      # 注：成员变量写入：`self.grammar_queue[i].grammar_wait_ct +` 保存队列状态，后续 event loop 会持续消费或填充它。
 179:                 if (
      # 注：多行分支开始：完整条件在接下来几行，通常用于组合多个请求参数或运行时状态。
 180:                     self.grammar_queue[i].grammar_wait_ct
 181:                     >= self.SGLANG_GRAMMAR_MAX_POLL_ITERATIONS
 182:                 ):
 183:                     # Timeout after max poll iterations
 184:                     # The actual waiting time is SGLANG_GRAMMAR_MAX_POLL_ITERATIONS * max(SGLANG_GRAMMAR_POLL_INTERVAL, GPU_forward_batch_latency)
 185:                     failed_req_idxs.add(i)
      # 注：对象/库方法调用：`failed_req_idxs.add` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 186: 
 187:         # Sync ready and failed requests across all ranks
 188:         if self.grammar_sync_size == 1:
      # 注：分支判断：根据请求的采样/grammar 约束 `self.grammar_sync_size == 1` 决定是否进入受限解码路径。
 189:             synced_ready_req_idxs = ready_req_idxs
      # 注：局部状态绑定：`synced_ready_req_idxs` 保存请求或 batch 状态，后续调度/执行会继续改写它。
 190:             synced_failed_req_idxs = failed_req_idxs
      # 注：局部状态绑定：`synced_failed_req_idxs` 保存请求或 batch 状态，后续调度/执行会继续改写它。
 191:         else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
 192:             all_gather_output = [None] * self.grammar_sync_size
      # 注：局部状态绑定：`all_gather_output` 保存本阶段的中间结果，后续几行通常会立即消费它。
 193:             torch.distributed.all_gather_object(
      # 注：关键调用：`torch.distributed.all_gather_object` - 跨 rank 同步 grammar ready/failed 集合，保证各 rank batch 一致。
 194:                 all_gather_output,
 195:                 (ready_req_idxs, failed_req_idxs),
 196:                 group=self.grammar_sync_group,
 197:             )
 198:             synced_ready_req_idxs = set.intersection(*[x[0] for x in all_gather_output])
      # 注：局部状态绑定：`synced_ready_req_idxs` 保存请求或 batch 状态，后续调度/执行会继续改写它。
 199:             synced_failed_req_idxs = set.union(*[x[1] for x in all_gather_output])
      # 注：局部状态绑定：`synced_failed_req_idxs` 保存请求或 batch 状态，后续调度/执行会继续改写它。
 200: 
 201:         # Return ready requests
 202:         return_reqs: List[Req] = []
      # 注：字段/变量声明：`return_reqs` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
 203:         for i in synced_ready_req_idxs:
      # 注：循环处理：通常是在遍历请求、rank、worker、token 或候选项。
 204:             req = self.grammar_queue[i]
      # 注：局部状态绑定：`req` 保存请求或 batch 状态，后续调度/执行会继续改写它。
 205:             return_reqs.append(req)
      # 注：对象/库方法调用：`return_reqs.append` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 206:             if req.finished() or req.grammar is None:  # It is aborted by AbortReq
      # 注：分支判断：根据请求的采样/grammar 约束 `req.finished() or req.grammar is None:  # It is aborted by AbortReq` 决定是否进入受限解码路径。
 207:                 continue
      # 注：循环控制：跳过或结束当前循环轮次，通常代表这个请求/节点已被当前分支处理完。
 208: 
 209:             assert isinstance(req.grammar, futures.Future) and req.grammar_key
 210:             try:
      # 注：异常边界：下面的调用可能跨进程、I/O 或用户输入，失败时需要清理内部状态。
 211:                 req.grammar = req.grammar.result()
      # 注：对象状态写入：`req.grammar` 修改 `req` 的可见状态，读源码时要继续追踪它在哪里被消费。
 212:             except Exception as e:
      # 注：异常处理分支：把失败转换成可控清理、缓存失败对象或用户可见错误。
```

**读这段时抓住：**

- 单机时 ready set 可以直接使用；分布式时 ready 取交集，failed 取并集，偏保守保证各 rank 一致。
- 编译超时后会 cancel Future，并把失败对象写入 grammar backend cache。
- 这段解释了为什么 structured output 请求有时会先等待 grammar preprocessing，而不是立即进入 scheduler。


## XGrammar backend：token bitmask 路径

`XGrammarGrammarBackend` 使用 tokenizer 信息创建 `GrammarCompiler`，dispatch JSON/regex/EBNF 得到 compiled grammar。`XGrammarGrammar` 持有 `GrammarMatcher`，每步填 bitmask，然后在 GPU logits 上应用 bitmask。


In [ ]:
show_lines("python/sglang/srt/constrained/xgrammar_backend.py", 59, 130)
show_lines("python/sglang/srt/constrained/xgrammar_backend.py", 188, 245)


### `XGrammarGrammar`：matcher 状态如何变成 token bitmask

```python
# python/sglang/srt/constrained/xgrammar_backend.py:59-125
  59: class XGrammarGrammar(BaseGrammarObject):
      # 注：类定义：`XGrammarGrammar` 是这一段的状态/行为边界；先看字段，再看哪些方法会改字段。
  60: 
  61:     def __init__(
      # 注：函数定义：`__init__` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  62:         self,
  63:         matcher: GrammarMatcher,
  64:         vocab_size: int,
  65:         ctx: CompiledGrammar,
  66:         override_stop_tokens: Optional[Union[List[int], int]],
  67:         key_string: Optional[str] = None,
  68:         grammar_stats: Optional[GrammarStats] = GrammarStats(),
  69:     ) -> None:
  70:         super().__init__()
      # 注：函数/库调用：`super` 把当前阶段委托给外部 helper 或库实现。
  71:         self.matcher = matcher
      # 注：成员变量写入：`self.matcher` 会留在对象生命周期中，后续方法可能依赖这份状态。
  72:         self.vocab_size = vocab_size
      # 注：成员变量写入：`self.vocab_size` 会留在对象生命周期中，后续方法可能依赖这份状态。
  73:         self.ctx = ctx
      # 注：成员变量写入：`self.ctx` 会留在对象生命周期中，后续方法可能依赖这份状态。
  74:         self.override_stop_tokens = override_stop_tokens
      # 注：成员变量写入：`self.override_stop_tokens` 保存 token 相关状态，后续长度计算、cache key 或采样都会读取它。
  75:         self.accepted_tokens = []
      # 注：成员变量写入：`self.accepted_tokens` 保存 token 相关状态，后续长度计算、cache key 或采样都会读取它。
  76:         self.key_string = key_string
      # 注：成员变量写入：`self.key_string` 会留在对象生命周期中，后续方法可能依赖这份状态。
  77:         self.grammar_stats = grammar_stats
      # 注：成员变量写入：`self.grammar_stats` 会留在对象生命周期中，后续方法可能依赖这份状态。
  78: 
  79:     def accept_token(self, token: int):
      # 注：函数定义：`accept_token` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  80:         if not self.is_terminated():
      # 注：分支判断：只有满足 `not self.is_terminated()` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
  81:             self.current_token = token
      # 注：成员变量写入：`self.current_token` 保存 token 相关状态，后续长度计算、cache key 或采样都会读取它。
  82:             accepted = self.matcher.accept_token(token)
      # 注：局部状态绑定：`accepted` 保存本阶段的中间结果，后续几行通常会立即消费它。
  83:             if not accepted:
      # 注：分支判断：只有满足 `not accepted` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
  84:                 # log for debugging
  85:                 raise ValueError(
      # 注：函数/库调用：`ValueError` 把当前阶段委托给外部 helper 或库实现。
  86:                     f"Tokens not accepted: {token}\n"
  87:                     f"Accepted tokens: {self.accepted_tokens}\n"
  88:                     f"Key string: {self.key_string}"
  89:                 )
  90:             else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
  91:                 self.accepted_tokens.append(token)
      # 注：成员函数调用：`self.accepted_tokens.append` 会读取或更新当前对象状态，建议继续查看该方法定义。
  92: 
  93:     def rollback(self, k: int):
      # 注：函数定义：`rollback` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  94:         self.matcher.rollback(k)
      # 注：成员函数调用：`self.matcher.rollback` 会读取或更新当前对象状态，建议继续查看该方法定义。
  95:         self.accepted_tokens = self.accepted_tokens[:-k]
      # 注：成员变量写入：`self.accepted_tokens` 保存 token 相关状态，后续长度计算、cache key 或采样都会读取它。
  96: 
  97:     def is_terminated(self):
      # 注：函数定义：`is_terminated` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  98:         return self.matcher.is_terminated()
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
  99: 
 100:     def allocate_vocab_mask(
      # 注：函数定义：`allocate_vocab_mask` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 101:         self, vocab_size: int, batch_size: int, device
 102:     ) -> torch.Tensor:
 103:         return allocate_token_bitmask(batch_size, vocab_size)
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
 104: 
 105:     def fill_vocab_mask(self, vocab_mask: torch.Tensor, idx: int) -> None:
      # 注：函数定义：`fill_vocab_mask` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 106:         self.matcher.fill_next_token_bitmask(vocab_mask, idx)
      # 注：关键调用：`self.matcher.fill_next_token_bitmask` - 把当前 grammar 状态下的合法 token 写入 bitmask。
 107: 
 108:     @staticmethod
 109:     def move_vocab_mask(vocab_mask: torch.Tensor, device) -> torch.Tensor:
      # 注：函数定义：`move_vocab_mask` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 110:         return vocab_mask.to(device, non_blocking=True)
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
 111: 
 112:     def apply_vocab_mask(self, logits: torch.Tensor, vocab_mask: torch.Tensor) -> None:
      # 注：函数定义：`apply_vocab_mask` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 113:         if logits.device.type in {"cuda", "xpu", "musa"}:
      # 注：分支判断：只有满足 `logits.device.type in {"cuda", "xpu", "musa"}` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 114:             if _is_hip:
      # 注：分支判断：只有满足 `_is_hip` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 115:                 apply_token_bitmask_inplace_cuda(logits, vocab_mask)
      # 注：函数/库调用：`apply_token_bitmask_inplace_cuda` 把当前阶段委托给外部 helper 或库实现。
 116:             else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
 117:                 apply_token_bitmask_inplace_triton(logits, vocab_mask)
      # 注：关键调用：`apply_token_bitmask_inplace_triton` - 在 GPU logits 上原地应用 token mask，非法 token 会被屏蔽。
 118:         elif logits.device.type == "npu":
      # 注：补充分支：前面的条件不满足时，再检查 `logits.device.type == "npu"`。
 119:             import sgl_kernel_npu  # noqa: F401
 120: 
 121:             torch.ops.npu.apply_token_bitmask(logits, vocab_mask)
      # 注：对象/库方法调用：`torch.ops.npu.apply_token_bitmask` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
 122:         else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
 123:             raise RuntimeError(f"Unsupported device: {logits.device.type}")
      # 注：函数/库调用：`RuntimeError` 把当前阶段委托给外部 helper 或库实现。
 124: 
 125:     def copy(self):
      # 注：函数定义：`copy` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
```

**读这段时抓住：**

- `GrammarMatcher.accept_token` 是 xgrammar 的状态推进；SGLang 额外记录 accepted tokens 方便报错和 rollback。
- `fill_next_token_bitmask` 把下一步合法 token 写入 bitmask；后续 logits kernel 只看这个 bitmask。
- `apply_vocab_mask` 按设备选择 Triton/CUDA/NPU kernel，因此 CPU-only 环境适合读/编译，不适合执行 mask kernel。


### `XGrammarGrammarBackend`：tokenizer 信息决定 grammar 能否编译

```python
# python/sglang/srt/constrained/xgrammar_backend.py:188-245
 188: class XGrammarGrammarBackend(BaseGrammarBackend):
      # 注：类定义：`XGrammarGrammarBackend` 是这一段的状态/行为边界；先看字段，再看哪些方法会改字段。
 189:     def __init__(
      # 注：函数定义：`__init__` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 190:         self,
 191:         tokenizer,
 192:         vocab_size: int,
 193:         model_eos_token_ids: Optional[List[int]] = None,
 194:         any_whitespace: bool = True,
 195:     ):
 196:         super().__init__()
      # 注：函数/库调用：`super` 把当前阶段委托给外部 helper 或库实现。
 197: 
 198:         if hasattr(tokenizer, "init_xgrammar"):
      # 注：分支判断：根据请求的采样/grammar 约束 `hasattr(tokenizer, "init_xgrammar")` 决定是否进入受限解码路径。
 199:             # For special tokenizer
 200:             tokenizer_info, override_stop_tokens = tokenizer.init_xgrammar()
      # 注：局部状态绑定：`tokenizer_info, override_stop_tokens` 保存本阶段的中间结果，后续几行通常会立即消费它。
 201: 
 202:             if tokenizer_info is None:
      # 注：分支判断：只有满足 `tokenizer_info is None` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 203:                 # Not supported tokenizer
 204:                 raise TokenizerNotSupportedError(
      # 注：函数/库调用：`TokenizerNotSupportedError` 把当前阶段委托给外部 helper 或库实现。
 205:                     f"Tokenizer type {type(tokenizer).__name__} is not supported by XGrammar"
      # 注：函数/库调用：`type` 把当前阶段委托给外部 helper 或库实现。
 206:                 )
 207:         else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
 208:             # Create TokenizerInfo with model's EOS tokens as the authoritative stop tokens
 209:             # This ensures consistency between what the model considers EOS and what XGrammar uses
 210:             try:
      # 注：异常边界：下面的调用可能跨进程、I/O 或用户输入，失败时需要清理内部状态。
 211:                 tokenizer_info = TokenizerInfo.from_huggingface(
      # 注：局部状态绑定：`tokenizer_info` 保存本阶段的中间结果，后续几行通常会立即消费它。
 212:                     tokenizer, vocab_size=vocab_size, stop_token_ids=model_eos_token_ids
      # 注：局部状态绑定：`tokenizer, vocab_size` 保存本阶段的中间结果，后续几行通常会立即消费它。
 213:                 )
 214:                 override_stop_tokens = None
      # 注：局部状态绑定：`override_stop_tokens` 保存本阶段的中间结果，后续几行通常会立即消费它。
 215:             except Exception as e:
      # 注：异常处理分支：把失败转换成可控清理、缓存失败对象或用户可见错误。
 216:                 raise TokenizerNotSupportedError(
      # 注：函数/库调用：`TokenizerNotSupportedError` 把当前阶段委托给外部 helper 或库实现。
 217:                     f"Failed to create XGrammar TokenizerInfo from tokenizer: {e}"
 218:                 )
 219: 
 220:         self.grammar_compiler = GrammarCompiler(tokenizer_info=tokenizer_info)
      # 注：成员变量写入：`self.grammar_compiler` 会留在对象生命周期中，后续方法可能依赖这份状态。
 221:         self.vocab_size = vocab_size
      # 注：成员变量写入：`self.vocab_size` 会留在对象生命周期中，后续方法可能依赖这份状态。
 222:         self.override_stop_tokens = override_stop_tokens
      # 注：成员变量写入：`self.override_stop_tokens` 保存 token 相关状态，后续长度计算、cache key 或采样都会读取它。
 223:         self.any_whitespace = any_whitespace
      # 注：成员变量写入：`self.any_whitespace` 会留在对象生命周期中，后续方法可能依赖这份状态。
 224: 
 225:     @property
 226:     def is_support_token_filter(self):
      # 注：函数定义：`is_support_token_filter` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 227:         return True
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
 228: 
 229:     @staticmethod
 230:     def allocate_vocab_mask(vocab_size: int, batch_size: int, device) -> torch.Tensor:
      # 注：函数定义：`allocate_vocab_mask` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 231:         return allocate_token_bitmask(batch_size, vocab_size)
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
 232: 
 233:     @staticmethod
 234:     def move_vocab_mask(vocab_mask: torch.Tensor, device) -> torch.Tensor:
      # 注：函数定义：`move_vocab_mask` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 235:         return vocab_mask.to(device, non_blocking=True)
      # 注：返回出口：把本阶段整理出的状态交给调用方；读调用链时要回到上层看它如何被消费。
 236: 
 237:     @staticmethod
 238:     def apply_vocab_mask(logits: torch.Tensor, vocab_mask: torch.Tensor) -> None:
      # 注：函数定义：`apply_vocab_mask` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
 239:         if logits.device.type in {"cuda", "npu", "xpu", "musa"}:
      # 注：分支判断：只有满足 `logits.device.type in {"cuda", "npu", "xpu", "musa"}` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 240:             if _is_hip:
      # 注：分支判断：只有满足 `_is_hip` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 241:                 apply_token_bitmask_inplace_cuda(logits, vocab_mask)
      # 注：函数/库调用：`apply_token_bitmask_inplace_cuda` 把当前阶段委托给外部 helper 或库实现。
 242:             else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
 243:                 apply_token_bitmask_inplace_triton(logits, vocab_mask)
      # 注：关键调用：`apply_token_bitmask_inplace_triton` - 在 GPU logits 上原地应用 token mask，非法 token 会被屏蔽。
 244:         else:
      # 注：兜底分支：前面的 if/elif 都不成立时进入，常代表默认模式或降级路径。
 245:             raise RuntimeError(f"Unsupported device: {logits.device.type}")
      # 注：函数/库调用：`RuntimeError` 把当前阶段委托给外部 helper 或库实现。
```

**读这段时抓住：**

- backend 初始化时先把 Hugging Face tokenizer 转成 `TokenizerInfo`，失败则可能降级为 grammar_backend=none。
- `model_eos_token_ids` 被传入 tokenizer info，确保 grammar stop token 与模型 EOS 语义一致。
- `is_support_token_filter` 返回 True，strict thinking 等功能会检查这个能力。


## Compressed FSM / Jump-forward 是什么

在 regex/FSM 约束里，某些状态后面只有一条确定路径。例如 schema 中固定的字段名、标点、引号。逐 token 生成这些确定片段会浪费 decode 步。

SGLang 的 jump-forward 思路是：如果当前 FSM 状态能确定接下来的一段字符串，就直接生成这段字符串，再重新 tokenization / 更新 grammar 状态。`outlines_jump_forward.py` 中的注释直接写着 “Faster constrained decoding with jump forward decoding / compressed finite state machine.”


In [ ]:
show_lines("python/sglang/srt/constrained/outlines_jump_forward.py", 1, 40)
show_lines("python/sglang/srt/constrained/outlines_jump_forward.py", 144, 176)


### `OutlinesJumpForwardMap`：把确定性 FSM 边压缩成可跳过片段

```python
# python/sglang/srt/constrained/outlines_jump_forward.py:62-110
  62: def init_state_to_jump_forward(regex_string):
      # 注：函数定义：`init_state_to_jump_forward` 是调用边界；参数决定它从上游接收哪些状态，返回值决定下游能看到什么。
  63:     try:
      # 注：异常边界：下面的调用可能跨进程、I/O 或用户输入，失败时需要清理内部状态。
  64:         regex_pattern = interegular.parse_pattern(regex_string)
      # 注：局部状态绑定：`regex_pattern` 保存本阶段的中间结果，后续几行通常会立即消费它。
  65:     except InvalidSyntax as e:
      # 注：异常处理分支：把失败转换成可控清理、缓存失败对象或用户可见错误。
  66:         logger.warning(f"skip invalid regex: {regex_string}, {e=}")
      # 注：对象/库方法调用：`logger.warning` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
  67:         return
      # 注：返回出口：提前结束当前路径，通常表示这个分支已经完成处理或无需继续推进。
  68: 
  69:     byte_fsm = make_byte_level_fsm(regex_pattern.to_fsm().reduce(), keep_utf8=True)
      # 注：局部状态绑定：`byte_fsm` 保存本阶段的中间结果，后续几行通常会立即消费它。
  70:     regex_fsm, _ = make_deterministic_fsm(byte_fsm)
      # 注：局部状态绑定：`regex_fsm, _` 保存本阶段的中间结果，后续几行通常会立即消费它。
  71: 
  72:     fsm_info: FSMInfo = regex_fsm.fsm_info
      # 注：字段/变量声明：`fsm_info` 带有类型信息；它描述这个对象或阶段会保存的状态形状。
  73: 
  74:     symbol_to_id = fsm_info.alphabet_symbol_mapping
      # 注：局部状态绑定：`symbol_to_id` 保存本阶段的中间结果，后续几行通常会立即消费它。
  75:     id_to_symbol = {}
      # 注：局部状态绑定：`id_to_symbol` 保存本阶段的中间结果，后续几行通常会立即消费它。
  76:     for symbol, id_ in symbol_to_id.items():
      # 注：循环处理：通常是在遍历请求、rank、worker、token 或候选项。
  77:         id_to_symbol.setdefault(id_, []).append(symbol)
      # 注：对象/库方法调用：`id_to_symbol.setdefault` 把当前对象状态交给另一个组件处理，建议追踪该对象的生命周期。
  78: 
  79:     transitions = fsm_info.transitions
      # 注：局部状态绑定：`transitions` 保存本阶段的中间结果，后续几行通常会立即消费它。
  80: 
  81:     outgoings_ct = defaultdict(int)
      # 注：局部状态绑定：`outgoings_ct` 保存本阶段的中间结果，后续几行通常会立即消费它。
  82:     # NOTE(lsyin): Final states can lead to terminate, so they have one outgoing edge naturally
  83:     for s in fsm_info.finals:
      # 注：循环处理：通常是在遍历请求、rank、worker、token 或候选项。
  84:         outgoings_ct[s] = 1
      # 注：局部状态绑定：`outgoings_ct[s]` 保存本阶段的中间结果，后续几行通常会立即消费它。
  85: 
  86:     state_to_jump_forward = {}
      # 注：局部状态绑定：`state_to_jump_forward` 保存本阶段的中间结果，后续几行通常会立即消费它。
  87:     for (state, id_), next_state in transitions.items():
      # 注：循环处理：通常是在遍历请求、rank、worker、token 或候选项。
  88:         if id_ == fsm_info.alphabet_anything_value:
      # 注：分支判断：只有满足 `id_ == fsm_info.alphabet_anything_value` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
  89:             # Arbitrarily symbol cannot be recognized as jump forward
  90:             continue
      # 注：循环控制：跳过或结束当前循环轮次，通常代表这个请求/节点已被当前分支处理完。
  91: 
  92:         symbols = id_to_symbol[id_]
      # 注：局部状态绑定：`symbols` 保存本阶段的中间结果，后续几行通常会立即消费它。
  93:         for c in symbols:
      # 注：循环处理：通常是在遍历请求、rank、worker、token 或候选项。
  94:             if len(c) > 1:
      # 注：分支判断：只有满足 `len(c) > 1` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
  95:                 # Skip byte level transitions like c = "5E"
  96:                 continue
      # 注：循环控制：跳过或结束当前循环轮次，通常代表这个请求/节点已被当前分支处理完。
  97: 
  98:             outgoings_ct[state] += 1
      # 注：局部状态绑定：`outgoings_ct[state] +` 保存本阶段的中间结果，后续几行通常会立即消费它。
  99:             if outgoings_ct[state] > 1:
      # 注：分支判断：只有满足 `outgoings_ct[state] > 1` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 100:                 if state in state_to_jump_forward:
      # 注：分支判断：只有满足 `state in state_to_jump_forward` 时才进入该路径；这通常是在区分部署模式、请求类型或资源状态。
 101:                     del state_to_jump_forward[state]
 102:                 break
      # 注：循环控制：跳过或结束当前循环轮次，通常代表这个请求/节点已被当前分支处理完。
 103: 
 104:             state_to_jump_forward[state] = JumpEdge(
      # 注：局部状态绑定：`state_to_jump_forward[state]` 保存本阶段的中间结果，后续几行通常会立即消费它。
 105:                 symbol=c,
 106:                 symbol_next_state=next_state,
 107:             )
 108: 
 109:     # Process the byte level jump forward
 110:     outgoings_ct = defaultdict(int)
      # 注：局部状态绑定：`outgoings_ct` 保存本阶段的中间结果，后续几行通常会立即消费它。
```

**读这段时抓住：**

- `init_state_to_jump_forward` 遍历 FSM 边，只保留那些从某状态出发没有歧义的确定边。
- 一旦发现同一 state 有多个可行 symbol/byte，就删除 jump-forward 记录，避免错误跳过采样。
- 这就是 compressed FSM 的直觉：把线性确定路径压缩成一次字符串推进。


## 小测试：检查当前环境可用的 grammar 后端

这个测试不启动模型。它检查依赖是否可 import，并定位 SGLang 注册逻辑。Mac 上如果没有 GPU，`apply_vocab_mask` 的 CUDA/Triton 路径不应执行，但编译侧依赖仍可检查。


In [ ]:
for mod in ["xgrammar", "outlines", "llguidance"]:
    try:
        __import__(mod)
        print(mod, "available")
    except Exception as e:
        print(mod, "not available:", type(e).__name__, str(e)[:120])

show_lines("python/sglang/srt/constrained/base_grammar_backend.py", 205, 270)


## 贡献者注意点

- Grammar object 必须可复制，因为缓存里保存的是“初始 grammar”，每个请求要拿自己的状态副本。
- Jump-forward 需要 retokenize，tokenizer 的 byte/string 边界非常容易出错。
- 多 rank 下 grammar ready/failed 的同步必须保守，否则不同 rank 会对同一 batch 使用不同约束。
- Reasoning model 的 strict thinking 会复用 grammar/token filter 机制，不要把 structured output 简化成“JSON only”。
